In [2]:
import rasterio
import numpy as np
import jenkspy
import geopandas as gpd
from rasterio.enums import Resampling
from rasterio.features import shapes
from shapely.geometry import shape
from scipy.ndimage import label
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION - MODIFY THESE PARAMETERS
# =============================================================================

# INPUT RASTER PATH
RASTER_PATH = r"C:\Users\resea\OneDrive\Desktop\STP\stp_sutability_24b3189d21324089b2a84d5324763f31_map.tif"  # ← CHANGE THIS PATH

# STP PARAMETERS
TREATMENT_TECHNOLOGY = "Membrane Bio Reactor (MBR)"  # Choose from technologies below
MLD_CAPACITY = 20.0  # Million Liters per Day
CUSTOM_LAND_PER_MLD = None  # Set to None to use default, or specify custom value (e.g., 0.08)

# OUTPUT PATH
OUTPUT_SHAPEFILE = r"C:\Users\resea\OneDrive\Desktop\STP\stp_suitable_sites_New.shp"  # Output shapefile path

# ANALYSIS PARAMETERS
N_CLASSES = 5  # Number of classification classes (not used in threshold mode)
TOP_N_CLUSTERS = 3  # Number of top clusters to export
USE_THRESHOLD_MODE = True  # Set to True to use direct threshold instead of classification
SUITABILITY_THRESHOLD = 0.353  # Minimum pixel value for high suitability
USE_FAST_CLASSIFICATION = True  # Only used if USE_THRESHOLD_MODE = False
MAX_SAMPLE_SIZE = 50000  # Only used if USE_THRESHOLD_MODE = False

# =============================================================================
# TECHNOLOGY OPTIONS
# =============================================================================
TECH_OPTIONS = {
    'Trickling Filter (TF)': 0.25,
    'Activated Sludge Process (ASP)': 0.15,
    'Sequential Batch Reactor (SBR)': 0.10,
    'BIOFOR-F': 0.08,
    'Extended Aeration (EA)': 0.15,
    'Membrane Bio Reactor (MBR)': 0.05
}

# =============================================================================
# CORE FUNCTIONS
# =============================================================================

def read_raster(path):
    """Read raster and extract metadata"""
    print(f" Reading raster: {path}")
    
    if not os.path.exists(path):
        raise FileNotFoundError(f"Raster file not found: {path}")
    
    with rasterio.open(path) as src:
        data = src.read(1, resampling=Resampling.nearest)
        profile = src.profile
        transform = src.transform
        crs = src.crs
        bounds = src.bounds
        nodata = src.nodata
        
        print(f" Raw data info:")
        print(f"   • NoData value: {nodata}")
        print(f"   • Raw data range: {data.min():.3e} - {data.max():.3e}")
        
        # Handle NoData values - convert extreme values to NaN
        if nodata is not None:
            data = np.where(data == nodata, np.nan, data)
        
        # Also handle extremely large negative values (common NoData representation)
        data = np.where(data < -1e10, np.nan, data)
        
        # Handle extremely large positive values
        data = np.where(data > 1e10, np.nan, data)
        
        # For suitability data, values should be between 0-1
        data = np.where((data < 0) | (data > 1), np.nan, data)
        
        # Calculate pixel size
        res_x = abs(transform[0])  # pixel width
        res_y = abs(transform[4])  # pixel height
        
        # Validate cleaned data
        valid_data = data[~np.isnan(data)]
        if len(valid_data) == 0:
            raise ValueError("Raster contains no valid data after cleaning")
        
        data_min, data_max = valid_data.min(), valid_data.max()
        valid_pixels = len(valid_data)
        total_pixels = data.size
        valid_percentage = (valid_pixels / total_pixels) * 100
        
        print(f" Raster loaded and cleaned successfully!")
        print(f"   • Size: {data.shape[1]} × {data.shape[0]} pixels")
        print(f"   • Resolution: {res_x:.1f}m × {res_y:.1f}m")
        print(f"   • Valid data range: {data_min:.3f} - {data_max:.3f}")
        print(f"   • Valid pixels: {valid_pixels:,} ({valid_percentage:.1f}%)")
        print(f"   • CRS: {crs}")
        
        return data, profile, res_x, res_y, transform, crs, bounds

def apply_threshold_classification(data, threshold=0.353):
    """Apply direct threshold classification (much faster than Jenks)"""
    print(f" Applying direct threshold classification...")
    print(f"   • Suitability threshold: {threshold}")
    
    # Remove NaN and invalid values
    valid_mask = ~np.isnan(data) & (data >= 0) & (data <= 1) & np.isfinite(data)
    valid_pixels = data[valid_mask]
    
    print(f" Data validation:")
    print(f"   • Total pixels: {data.size:,}")
    print(f"   • Valid pixels: {len(valid_pixels):,}")
    print(f"   • Valid percentage: {(len(valid_pixels)/data.size)*100:.1f}%")
    print(f"   • Valid data range: {valid_pixels.min():.6f} - {valid_pixels.max():.6f}")
    
    if len(valid_pixels) == 0:
        raise ValueError("No valid pixels found in the dataset")
    
    # Create binary classification: 1 = suitable (>= threshold), 0 = not suitable
    suitable_mask = (data >= threshold) & valid_mask
    
    # Count pixels above threshold
    suitable_pixels = np.sum(suitable_mask)
    suitable_percentage = (suitable_pixels / len(valid_pixels)) * 100
    
    print(f" Threshold analysis results:")
    print(f"   • Pixels above threshold ({threshold}): {suitable_pixels:,} ({suitable_percentage:.1f}%)")
    print(f"   • Pixels below threshold: {len(valid_pixels) - suitable_pixels:,} ({100-suitable_percentage:.1f}%)")
    
    if suitable_pixels == 0:
        raise ValueError(f"No pixels meet the suitability threshold of {threshold}")
    
    # Create a simple binary classification array (0 = not suitable, 1 = suitable)
    # Note: We'll use value 5 for suitable pixels to maintain compatibility with existing code
    reclassified = np.zeros_like(data, dtype=np.uint8)
    reclassified[suitable_mask] = 5  # Mark suitable pixels as class 5 (highest suitability)
    
    print(" Threshold classification completed!")
    
    return reclassified, threshold

def calculate_required_pixels(required_area_m2, res_x, res_y):
    """Calculate number of pixels needed for required area"""
    pixel_area = res_x * res_y
    pixels_needed = int(np.ceil(required_area_m2 / pixel_area))
    kernel_size = int(np.ceil(np.sqrt(pixels_needed)))
    return kernel_size, pixels_needed, pixel_area

def find_suitable_areas(reclassified, kernel_size, required_pixels, threshold_mode=True):
    """Find areas where all pixels meet suitability criteria"""
    if threshold_mode:
        print(f"🔍 Searching for areas where ALL pixels are above threshold...")
        print(f"   • Using {kernel_size}×{kernel_size} kernel")
        print(f"   • Required pixels per window: {required_pixels}")
    else:
        print(f"🔍 Searching for suitable areas with {kernel_size}×{kernel_size} kernel...")
    
    rows, cols = reclassified.shape
    suitable_mask = np.zeros_like(reclassified, dtype=np.uint8)
    
    # Calculate total number of windows
    total_windows = (rows - kernel_size + 1) * (cols - kernel_size + 1)
    
    print(f" Total windows to analyze: {total_windows:,}")
    
    # Progress bar for window analysis
    with tqdm(total=total_windows, desc="Analyzing windows", unit="windows") as pbar:
        for i in range(rows - kernel_size + 1):
            for j in range(cols - kernel_size + 1):
                window = reclassified[i:i+kernel_size, j:j+kernel_size]
                
                # Check if ALL pixels in window are suitable (value = 5)
                # and the total suitable pixels meet the area requirement
                if np.all(window == 5) and np.sum(window == 5) >= required_pixels:
                    suitable_mask[i:i+kernel_size, j:j+kernel_size] = 1
                
                pbar.update(1)
    
    suitable_pixels = np.sum(suitable_mask)
    print(f"✅ Found {suitable_pixels:,} suitable pixels")
    
    return suitable_mask

def extract_clusters_as_polygons(mask_array, transform, crs, min_area_m2=None):
    """Extract clusters and convert to polygons"""
    print(" Extracting clusters and converting to polygons...")
    
    # Label connected components
    labeled_array, num_features = label(mask_array)
    
    print(f" Found {num_features} connected components")
    
    if num_features == 0:
        return None
    
    # Convert raster clusters to polygons
    polygons = []
    areas = []
    
    print(" Converting raster clusters to vector polygons...")
    
    with tqdm(desc="Processing clusters", unit="cluster") as pbar:
        for geom, value in shapes(labeled_array.astype(np.uint8), transform=transform):
            if value > 0:  # Only process non-zero values
                poly = shape(geom)
                area_m2 = poly.area
                
                # Filter by minimum area if specified
                if min_area_m2 is None or area_m2 >= min_area_m2:
                    polygons.append(poly)
                    areas.append(area_m2)
                    pbar.update(1)
    
    if not polygons:
        print(" No clusters meet the minimum area requirement")
        return None
    
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame({
        'cluster_id': range(1, len(polygons) + 1),
        'area_m2': areas,
        'area_ha': [a/10000 for a in areas],
        'geometry': polygons
    }, crs=crs)
    
    # Sort by area (largest first)
    gdf = gdf.sort_values('area_m2', ascending=False).reset_index(drop=True)
    
    print(f"✅ Successfully extracted {len(gdf)} suitable clusters")
    
    return gdf

def display_results(clusters_gdf, required_area_ha, top_n=3):
    """Display analysis results"""
    if clusters_gdf is None or len(clusters_gdf) == 0:
        print("❌ No suitable areas found!")
        return False
    
    print(f"\n ANALYSIS RESULTS")
    print("=" * 50)
    
    num_clusters = len(clusters_gdf)
    top_clusters = clusters_gdf.head(top_n)
    
    print(f"✅ Found {num_clusters} suitable area(s)!")
    print(f"📋 Top {min(top_n, num_clusters)} clusters:")
    print()
    
    for idx, row in top_clusters.iterrows():
        print(f" Cluster {row['cluster_id']}:")
        print(f"   • Area: {row['area_ha']:.2f} ha ({row['area_m2']:,.0f} m²)")
        print(f"   • Size vs Required: {(row['area_ha']/required_area_ha)*100:.1f}%")
        print(f"   • Excess Area: {row['area_ha']-required_area_ha:.2f} ha")
        print()
    
    return True

def save_results(clusters_gdf, output_path, top_n=3):
    """Save results to shapefile"""
    if clusters_gdf is None or len(clusters_gdf) == 0:
        print(" No data to save")
        return False
    
    top_clusters = clusters_gdf.head(top_n)
    
    try:
        # Ensure output directory exists
        output_dir = os.path.dirname(output_path)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)
        
        # Save to shapefile
        top_clusters.to_file(output_path)
        print(f"✅ Results saved to: {output_path}")
        print(f"📁 Saved {len(top_clusters)} clusters")
        
        # Also save as CSV (without geometry)
        csv_path = output_path.replace('.shp', '.csv')
        top_clusters.drop('geometry', axis=1).to_csv(csv_path, index=False)
        print(f" Summary saved to: {csv_path}")
        
        return True
        
    except Exception as e:
        print(f" Error saving results: {str(e)}")
        return False

# =============================================================================
# MAIN ANALYSIS FUNCTION
# =============================================================================

def run_stp_analysis():
    """Main function to run STP site analysis"""
    
    print(" STP Site Suitability Analysis")
    print("=" * 50)
    
    # Validate inputs
    if not os.path.exists(RASTER_PATH):
        print(f" Error: Raster file not found: {RASTER_PATH}")
        print(" Please update the RASTER_PATH variable in the configuration section")
        return False
    
    if TREATMENT_TECHNOLOGY not in TECH_OPTIONS:
        print(f" Error: Unknown treatment technology: {TREATMENT_TECHNOLOGY}")
        print(f" Available options: {list(TECH_OPTIONS.keys())}")
        return False
    
    # Configuration summary
    land_per_mld = CUSTOM_LAND_PER_MLD if CUSTOM_LAND_PER_MLD is not None else TECH_OPTIONS[TREATMENT_TECHNOLOGY]
    required_area_ha = MLD_CAPACITY * land_per_mld
    required_area_m2 = required_area_ha * 10000
    
    print(" CONFIGURATION:")
    print(f"   • Technology: {TREATMENT_TECHNOLOGY}")
    print(f"   • Capacity: {MLD_CAPACITY} MLD")
    print(f"   • Land per MLD: {land_per_mld} ha/MLD")
    print(f"   • Required Area: {required_area_ha:.2f} ha ({required_area_m2:,.0f} m²)")
    if USE_THRESHOLD_MODE:
        print(f"   • Suitability Threshold: {SUITABILITY_THRESHOLD}")
        print(f"   • Mode: Direct Threshold (No Classification)")
    else:
        print(f"   • Mode: Classification-based Analysis")
    print()
    
    try:
        # Step 1: Read raster
        data, profile, res_x, res_y, transform, crs, bounds = read_raster(RASTER_PATH)
        
        # Step 2: Apply classification or threshold
        if USE_THRESHOLD_MODE:
            reclassified, threshold_info = apply_threshold_classification(data, SUITABILITY_THRESHOLD)
        else:
            reclassified, breaks = apply_jenks_classification(
                data, N_CLASSES, MAX_SAMPLE_SIZE, USE_FAST_CLASSIFICATION
            )
        
        # Step 3: Calculate spatial requirements
        kernel_size, required_pixels, pixel_area = calculate_required_pixels(
            required_area_m2, res_x, res_y
        )
        
        print(f"\n🔢 SPATIAL PARAMETERS:")
        print(f"   • Kernel Size: {kernel_size} × {kernel_size} pixels")
        print(f"   • Required Pixels: {required_pixels:,}")
        print(f"   • Pixel Area: {pixel_area:.1f} m²")
        print()
        
        # Step 4: Find suitable areas
        suitable_mask = find_suitable_areas(reclassified, kernel_size, required_pixels, USE_THRESHOLD_MODE)
        
        # Step 5: Extract clusters
        clusters_gdf = extract_clusters_as_polygons(
            suitable_mask, transform, crs, min_area_m2=required_area_m2
        )
        
        # Step 6: Display and save results
        success = display_results(clusters_gdf, required_area_ha, TOP_N_CLUSTERS)
        
        if success:
            save_results(clusters_gdf, OUTPUT_SHAPEFILE, TOP_N_CLUSTERS)
            print(f"\n Analysis completed successfully!")
            return True
        else:
            print(f"\n💡 SUGGESTIONS:")
            print("   • Try reducing the minimum capacity")
            print("   • Select a more compact technology (MBR, BIOFOR-F)")
            print("   • Consider accepting suitability class 4 instead of only class 5")
            return False
            
    except Exception as e:
        print(f"\n❌ Error during analysis: {str(e)}")
        return False

# =============================================================================
# RUN THE ANALYSIS
# =============================================================================

if __name__ == "__main__":
    # Run the analysis
    run_stp_analysis()

🏭 STP Site Suitability Analysis
⚙️ CONFIGURATION:
   • Technology: Membrane Bio Reactor (MBR)
   • Capacity: 20.0 MLD
   • Land per MLD: 0.05 ha/MLD
   • Required Area: 1.00 ha (10,000 m²)
   • Suitability Threshold: 0.353
   • Mode: Direct Threshold (No Classification)

📁 Reading raster: C:\Users\resea\OneDrive\Desktop\STP\stp_sutability_24b3189d21324089b2a84d5324763f31_map.tif
📊 Raw data info:
   • NoData value: -3.4028234663852886e+38
   • Raw data range: -3.403e+38 - 6.084e-01
✅ Raster loaded and cleaned successfully!
   • Size: 4454 × 1984 pixels
   • Resolution: 30.0m × 30.0m
   • Valid data range: 0.000 - 0.608
   • Valid pixels: 4,214,934 (47.7%)
   • CRS: EPSG:32644
🎯 Applying direct threshold classification...
   • Suitability threshold: 0.353
📊 Data validation:
   • Total pixels: 8,836,736
   • Valid pixels: 4,214,934
   • Valid percentage: 47.7%
   • Valid data range: 0.000000 - 0.608384
📊 Threshold analysis results:
   • Pixels above threshold (0.353): 1,469,379 (34.9%)
  

Analyzing windows: 100%|█████████████████████████████████████████████| 8817431/8817431 [02:23<00:00, 61615.32windows/s]


✅ Found 1,383,015 suitable pixels
🔄 Extracting clusters and converting to polygons...
📊 Found 995 connected components
🔄 Converting raster clusters to vector polygons...


Processing clusters: 992cluster [00:00, 1170.41cluster/s]


✅ Successfully extracted 992 suitable clusters

🎯 ANALYSIS RESULTS
✅ Found 992 suitable area(s)!
📋 Top 3 clusters:

📍 Cluster 699:
   • Area: 35978.31 ha (359,783,100 m²)
   • Size vs Required: 3597831.0%
   • Excess Area: 35977.31 ha

📍 Cluster 293:
   • Area: 31255.74 ha (312,557,400 m²)
   • Size vs Required: 3125574.0%
   • Excess Area: 31254.74 ha

📍 Cluster 98:
   • Area: 5454.45 ha (54,544,500 m²)
   • Size vs Required: 545445.0%
   • Excess Area: 5453.45 ha

✅ Results saved to: C:\Users\resea\OneDrive\Desktop\STP\stp_suitable_sites_New.shp
📁 Saved 3 clusters
📊 Summary saved to: C:\Users\resea\OneDrive\Desktop\STP\stp_suitable_sites_New.csv

🎉 Analysis completed successfully!
